Видалення даних таблиць. Перед початком (щоб уникнути помилки)

In [0]:
%sql

DROP TABLE IF EXISTS workspace.silver.lineitem;
DROP TABLE IF EXISTS workspace.silver.orders;
DROP TABLE IF EXISTS workspace.silver.partsupp;
DROP TABLE IF EXISTS workspace.silver.part;
DROP TABLE IF EXISTS workspace.silver.supplier;
DROP TABLE IF EXISTS workspace.silver.customer;
DROP TABLE IF EXISTS workspace.silver.nation;
DROP TABLE IF EXISTS workspace.silver.region;

SHOW TABLES IN workspace.silver;

Silver layer є очищеним і структурованим шаром даних між Bronze та Gold. У ньому таблиці TPC-H (region, nation, customer, supplier, part, partsupp, orders, lineitem) приводяться до третьої нормальної форми (3NF). Ми створюємо сутності і оперуємо ними, кожна таблиця окремо має свої дані до сутності (напртклад клієнт - тільки клієнт, інфромація про регіон, це окрема сутність регіон, інформація про те яка країна - інша сутність.)  щоб коли ми в своєму мінідовіднику міняємо інформацію все решта не поплило. 

Також тут перевірямп тип даних, та те щоб ключі працювали коректно. Внутрішні - кожен Primary Key - відповідав своїй одній одиниці, а Foreign Key відправляв до потрібноі одиниці в іншій таблиці. (наприклад таблиця з замовленнями містить ід користувача і ід продукту, а не всю інформацію про продукти і так далі)

Для забезпечення того щоб все не зламалося і не зіхало потрібний порядок створення таблиць. Спочатку ті від якої залежать і вони не посилаються на інші, а тоді вже ті які залежать. Так як ми хочемо звести все в 3nf форму. 


Найбільш незалежна таблиця REGION
далі буде порядок
- Nation (зележить від region)
- Customer(залежить від Nation)
- Orders (залежить від Customer)


(тут ми доходим до стелі де потрібно пройтися по інших таблицях від найменш залежної)
- part (незалежна)
- supplier (залежить від part)
- partsupp (залежить від part + nation)

кінцева звязка
partsupp + orders -> lineitem


** тест на цікавість, чи нема цих звязків на бронз**

In [0]:
%sql
SHOW TABLES IN workspace.bronze;

In [0]:
%sql
SELECT COUNT(*) AS broken_links
FROM workspace.bronze.customer c
LEFT JOIN workspace.bronze.nation n
    ON c.c_nationkey = n.n_nationkey
WHERE n.n_nationkey IS NULL;

In [0]:
%sql
SELECT
    constraint_name,
    table_name,
    constraint_type
FROM workspace.information_schema.table_constraints
WHERE table_schema = 'bronze';

структура узгоджена (назви одинакові) але самих звязків як таких немає 

1. REGION

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.region AS
SELECT
    CAST(r_regionkey AS BIGINT) AS r_regionkey,
    TRIM(r_name) AS r_name,
    TRIM(r_comment) AS r_comment
FROM workspace.bronze.region
WHERE r_regionkey IS NOT NULL;

Перевірки:

чи не змінилася кількість рядків

In [0]:
%sql
SELECT
    (SELECT COUNT(*) FROM workspace.bronze.region) AS bronze_count,
    (SELECT COUNT(*) FROM workspace.silver.region) AS silver_count;

In [0]:
%sql
SELECT COUNT(*) AS null_region_keys
FROM workspace.silver.region
WHERE r_regionkey IS NULL;

перевірка дублів у ключах

In [0]:
%sql
SELECT
    r_regionkey,
    COUNT(*) AS cnt
FROM workspace.silver.region
GROUP BY r_regionkey
HAVING COUNT(*) > 1;

2. NATION

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.nation AS
SELECT
    CAST(n_nationkey AS BIGINT) AS n_nationkey,
    TRIM(n_name) AS n_name,
    CAST(n_regionkey AS BIGINT) AS n_regionkey,
    TRIM(n_comment) AS n_comment
FROM workspace.bronze.nation
WHERE n_nationkey IS NOT NULL
  AND n_regionkey IS NOT NULL;

Перевірка

In [0]:
%sql
SELECT COUNT(*) AS null_keys
FROM workspace.silver.nation
WHERE n_nationkey IS NULL
   OR n_regionkey IS NULL;

In [0]:
%sql
SELECT
    n_nationkey,
    COUNT(*) AS cnt
FROM workspace.silver.nation
GROUP BY n_nationkey
HAVING COUNT(*) > 1;

Перевірка зовнішнього ключа

In [0]:
%sql
SELECT COUNT(*) AS broken_links
FROM workspace.silver.nation n
LEFT JOIN workspace.silver.region r
    ON n.n_regionkey = r.r_regionkey
WHERE r.r_regionkey IS NULL;

3. CUSTOMER

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.customer AS
SELECT
    CAST(c_custkey AS BIGINT) AS c_custkey,
    TRIM(c_name) AS c_name,
    TRIM(c_address) AS c_address,
    CAST(c_nationkey AS BIGINT) AS c_nationkey,
    TRIM(c_phone) AS c_phone,
    CAST(c_acctbal AS DECIMAL(18,2)) AS c_acctbal,
    TRIM(c_mktsegment) AS c_mktsegment,
    TRIM(c_comment) AS c_comment,
    _ingested_at
FROM workspace.bronze.customer
WHERE c_custkey IS NOT NULL
  AND c_nationkey IS NOT NULL;

Перевірки

In [0]:
%sql
SELECT c_custkey, COUNT(*) AS cnt
FROM workspace.silver.customer
GROUP BY c_custkey
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT c.c_custkey, c.c_nationkey
FROM workspace.silver.customer c
LEFT JOIN workspace.silver.nation n
    ON c.c_nationkey = n.n_nationkey
WHERE n.n_nationkey IS NULL;

4. ORDERS


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.orders AS
SELECT
    CAST(o_orderkey AS BIGINT) AS o_orderkey,
    CAST(o_custkey AS BIGINT) AS o_custkey,
    TRIM(o_orderstatus) AS o_orderstatus,
    CAST(o_totalprice AS DECIMAL(18,2)) AS o_totalprice,
    CAST(o_orderdate AS DATE) AS o_orderdate,
    TRIM(o_orderpriority) AS o_orderpriority,
    TRIM(o_clerk) AS o_clerk,
    TRIM(o_shippriority) AS o_shippriority,
    TRIM(o_comment) AS o_comment,
    _ingested_at
FROM workspace.bronze.orders
WHERE o_orderkey IS NOT NULL
  AND o_custkey IS NOT NULL;

перевірки

In [0]:
%sql
SELECT o_orderkey, COUNT(*) AS cnt
FROM workspace.silver.orders
GROUP BY o_orderkey
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT o.o_orderkey, o.o_custkey
FROM workspace.silver.orders o
LEFT JOIN workspace.silver.customer c
    ON o.o_custkey = c.c_custkey
WHERE c.c_custkey IS NULL;

5. PART

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.part AS
SELECT
    CAST(p_partkey AS BIGINT) AS p_partkey,
    TRIM(p_name) AS p_name,
    TRIM(p_mfgr) AS p_mfgr,
    TRIM(p_brand) AS p_brand,
    TRIM(p_type) AS p_type,
    CAST(p_size AS INT) AS p_size,
    TRIM(p_container) AS p_container,
    CAST(p_retailprice AS DECIMAL(18,2)) AS p_retailprice,
    TRIM(p_comment) AS p_comment,
    _ingested_at
FROM workspace.bronze.part
WHERE p_partkey IS NOT NULL;

перевірка

In [0]:
%sql
SELECT p_partkey, COUNT(*) AS cnt
FROM workspace.silver.part
GROUP BY p_partkey
HAVING COUNT(*) > 1;

6. SUPPLIER

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.supplier AS
SELECT
    CAST(s_suppkey AS BIGINT) AS s_suppkey,
    TRIM(s_name) AS s_name,
    TRIM(s_address) AS s_address,
    CAST(s_nationkey AS BIGINT) AS s_nationkey,
    TRIM(s_phone) AS s_phone,
    CAST(s_acctbal AS DECIMAL(18,2)) AS s_acctbal,
    TRIM(s_comment) AS s_comment,
    _ingested_at
FROM workspace.bronze.supplier
WHERE s_suppkey IS NOT NULL
  AND s_nationkey IS NOT NULL;

тест

In [0]:
%sql
SELECT s.s_suppkey, s.s_nationkey
FROM workspace.silver.supplier s
LEFT JOIN workspace.silver.nation n
    ON s.s_nationkey = n.n_nationkey
WHERE n.n_nationkey IS NULL;

7. PARTSUPP

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.partsupp AS
SELECT
    CAST(ps_partkey AS BIGINT) AS ps_partkey,
    CAST(ps_suppkey AS BIGINT) AS ps_suppkey,
    CAST(ps_availqty AS INT) AS ps_availqty,
    CAST(ps_supplycost AS DECIMAL(18,2)) AS ps_supplycost,
    TRIM(ps_comment) AS ps_comment,
    _ingested_at
FROM workspace.bronze.partsupp
WHERE ps_partkey IS NOT NULL
  AND ps_suppkey IS NOT NULL;

Перевірки

In [0]:
%sql
SELECT
    ps_partkey,
    ps_suppkey,
    COUNT(*) AS cnt
FROM workspace.silver.partsupp
GROUP BY ps_partkey, ps_suppkey
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT ps.ps_partkey
FROM workspace.silver.partsupp ps
LEFT JOIN workspace.silver.part p
    ON ps.ps_partkey = p.p_partkey
WHERE p.p_partkey IS NULL;

7. PARTSUPP

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.partsupp AS
SELECT
    CAST(ps_partkey AS BIGINT) AS ps_partkey,
    CAST(ps_suppkey AS BIGINT) AS ps_suppkey,
    CAST(ps_availqty AS INT) AS ps_availqty,
    CAST(ps_supplycost AS DECIMAL(18,2)) AS ps_supplycost,
    TRIM(ps_comment) AS ps_comment,
    _ingested_at
FROM workspace.bronze.partsupp
WHERE ps_partkey IS NOT NULL
  AND ps_suppkey IS NOT NULL;

Перевірка

In [0]:
%sql
SELECT
    ps_partkey,
    ps_suppkey,
    COUNT(*) AS cnt
FROM workspace.silver.partsupp
GROUP BY ps_partkey, ps_suppkey
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT ps.ps_partkey
FROM workspace.silver.partsupp ps
LEFT JOIN workspace.silver.part p
    ON ps.ps_partkey = p.p_partkey
WHERE p.p_partkey IS NULL;

In [0]:
%sql
SELECT ps.ps_suppkey
FROM workspace.silver.partsupp ps
LEFT JOIN workspace.silver.supplier s
    ON ps.ps_suppkey = s.s_suppkey
WHERE s.s_suppkey IS NULL;

8. LINEITEM

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.lineitem AS
SELECT
    CAST(l_orderkey AS BIGINT) AS l_orderkey,
    CAST(l_partkey AS BIGINT) AS l_partkey,
    CAST(l_suppkey AS BIGINT) AS l_suppkey,
    CAST(l_linenumber AS INT) AS l_linenumber,
    CAST(l_quantity AS DECIMAL(18,2)) AS l_quantity,
    CAST(l_extendedprice AS DECIMAL(18,2)) AS l_extendedprice,
    CAST(l_discount AS DECIMAL(18,2)) AS l_discount,
    CAST(l_tax AS DECIMAL(18,2)) AS l_tax,
    TRIM(l_returnflag) AS l_returnflag,
    TRIM(l_linestatus) AS l_linestatus,
    CAST(l_shipdate AS DATE) AS l_shipdate,
    CAST(l_commitdate AS DATE) AS l_commitdate,
    CAST(l_receiptdate AS DATE) AS l_receiptdate,
    TRIM(l_shipinstruct) AS l_shipinstruct,
    TRIM(l_shipmode) AS l_shipmode,
    TRIM(l_comment) AS l_comment,
    _ingested_at
FROM workspace.bronze.lineitem
WHERE l_orderkey IS NOT NULL
  AND l_partkey IS NOT NULL
  AND l_suppkey IS NOT NULL
  AND l_linenumber IS NOT NULL;

перевірка

In [0]:
%sql
SELECT
    l_orderkey,
    l_linenumber,
    COUNT(*) AS cnt
FROM workspace.silver.lineitem
GROUP BY l_orderkey, l_linenumber
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT COUNT(*) AS broken_links
FROM workspace.silver.lineitem l
LEFT JOIN workspace.silver.partsupp ps
    ON l.l_partkey = ps.ps_partkey
   AND l.l_suppkey = ps.ps_suppkey
WHERE ps.ps_partkey IS NULL;